# 1주차 보충 · 배열의 축, 행렬곱과 벡터 검색

두 픽셀의 밝기를 [20, 80]으로 나타내면 평균 밝기는 50이고, 오른쪽에서 왼쪽을 뺀 밝기 차는 60입니다. 여러 이미지에 같은 계산을 적용할 때는 입력 특성을 열로, 계산할 항목을 계수 행렬의 열로 놓습니다. 이 장에서는 행렬곱의 한 칸을 직접 계산하고, 그 방법으로 질의와 후보 사이의 점수표를 만듭니다. 각 후보를 구별하도록 붙인 고유 값을 **식별자(identifier, ID)**라고 합니다. 후보를 재배열하면 열 위치는 바뀌어도 그 후보의 ID는 그대로입니다.

| 학습 내용 | 예제와 연습 |
|---|---|
| 배열의 행·열, 인덱스와 특성, 행렬곱 | 1–3절과 연습 A |
| 벡터의 내적·코사인 유사도와 후보 ID, 토큰 벡터의 평균 | 5절과 연습 B |
| 여러 배열의 결합 | 4절과 연습 C |

1절, 4절, 5절과 연습 A·B는 각각 첫 코드 셀에서 NumPy를 불러옵니다. 새 커널에서 해당 절의 코드 셀을 위부터 실행하세요. 아래 픽셀값, 벡터와 ID는 손계산을 위해 만든 연습 데이터입니다. 국제대회에서 실제로 제공한 데이터와 요구한 출력은 5절과 마지막 원문 문제에서 확인합니다.


<a id="bridge-axes"></a>

## 1. 2차원 배열의 행, 열과 인덱스

아래 표는 가로로 두 픽셀인 회색조 이미지 세 개를 나타냅니다. 두 열의 값은 0(검정)부터 255(흰색)까지입니다. **이미지 ID(식별자)**는 이 표에서 이미지를 구별하는 고유 번호입니다. 행을 다시 정렬해도 ID 52는 같은 이미지를 가리킵니다. 계산에 쓰는 입력값을 **특성(feature)**이라고 합니다. 이 예에서 두 특성은 왼쪽 픽셀 밝기와 오른쪽 픽셀 밝기입니다.

| 이미지 ID | 왼쪽 픽셀 | 오른쪽 픽셀 |
|---:|---:|---:|
| 17 | 20 | 80 |
| 52 | 60 | 30 |
| 8 | 40 | 120 |

수를 행과 열에 직사각형으로 배열한 것을 **행렬**이라고 합니다. 여기서는 두 픽셀 열을 NumPy의 2차원 배열 <code>X</code>에 넣어 행렬로 사용하고, 이미지 ID는 <code>image_ids</code>에 따로 둡니다. <code>X.shape</code>는 각 축의 길이를 차례로 나타내므로 <code>(3, 2)</code>입니다. 첫째 축(axis 0)을 따라 움직이면 이미지가 바뀌고, 둘째 축(axis 1)을 따라 움직이면 픽셀 위치가 바뀝니다.

**인덱스**는 0부터 센 위치입니다. <code>X[1, 0]</code>은 두 번째 이미지의 왼쪽 픽셀 60입니다. 같은 행의 ID는 <code>image_ids[1]</code>으로 찾습니다. <code>X[1, :]</code>와 <code>X[:, 0]</code>에서 콜론은 해당 축의 모든 위치를 뜻합니다. 두 식의 결과를 각각 예상해 보세요.


In [1]:
import numpy as np

image_ids = np.array([17, 52, 8])
X = np.array([[20, 80], [60, 30], [40, 120]])
print("X.shape =", X.shape, "/ 첫 행:", X[0])
print("ID", image_ids[1], "의 두 픽셀:", X[1, :])
print("각 이미지의 왼쪽 픽셀:", X[:, 0])
print("X[1, 0] =", X[1, 0])
assert X.shape == (3, 2) and X[1, 0] == 60


X.shape = (3, 2) / 첫 행: [20 80]
ID 52 의 두 픽셀: [60 30]
각 이미지의 왼쪽 픽셀: [20 60 40]
X[1, 0] = 60


### 한 벡터와 계수 벡터의 내적

<code>X[1, :]</code>는 한 이미지의 두 특성 <code>[60, 30]</code>이고, <code>X[:, 0]</code>은 세 이미지의 왼쪽 밝기 <code>[20, 60, 40]</code>입니다. 각각의 shape는 <code>(2,)</code>와 <code>(3,)</code>입니다.

이미지 하나의 평균 밝기는 <code>0.5 × 왼쪽 + 0.5 × 오른쪽</code>입니다. 첫 행 <code>x = [20, 80]</code>과 계수 <code>w = [0.5, 0.5]</code>를 같은 순서로 놓습니다.

| 계산 | 결과 | 뜻 |
|---|---|---|
| <code>x * w</code> | [10, 40] | 각 위치의 곱 |
| <code>x @ w</code> | 50 | 두 곱의 합, 즉 평균 밝기 |

길이가 d인 두 벡터의 **내적**은 <code>x[0]w[0] + … + x[d−1]w[d−1]</code>입니다. 식에서 붙여 쓴 두 수는 곱합니다. 계수를 <code>[-1, 1]</code>로 바꾸면 어떤 밝기 차를 구하는지 첫 행에서 계산해 보세요.


In [2]:
first_image = X[0]
average_weights = np.array([0.5, 0.5])
print("좌표별 곱:", first_image * average_weights)
print("평균 밝기:", first_image @ average_weights)
print("오른쪽 − 왼쪽:", first_image @ np.array([-1, 1]))
assert first_image @ average_weights == 50


좌표별 곱: [10. 40.]
평균 밝기: 50.0
오른쪽 − 왼쪽: 60


## 2. 행렬곱과 선형 변환

각 이미지에서 **평균 밝기**와 **오른쪽 밝기 − 왼쪽 밝기**를 이 순서로 구합니다. 왼쪽·오른쪽 밝기를 각각 L, R이라 하면 두 식은 <code>(L + R)/2</code>와 <code>R − L</code>입니다. 각 식의 계수를 한 열에 모으면 다음 행렬 W가 됩니다.

| W의 행 | 평균 밝기 | 오른쪽 − 왼쪽 |
|---|---:|---:|
| 왼쪽 픽셀 L | 0.5 | −1 |
| 오른쪽 픽셀 R | 0.5 | 1 |

<code>Y = X @ W</code>의 <code>Y[1, 1]</code>을 구할 때는 X의 1번 **행** <code>[60, 30]</code>과 W의 1번 **열** <code>[-1, 1]</code>을 택합니다. 같은 픽셀에 해당하는 수끼리 곱하면 <code>60×(−1) + 30×1 = −30</code>입니다. 음수는 이 이미지의 오른쪽이 왼쪽보다 어둡다는 뜻입니다. ID 8의 평균과 밝기 차도 손으로 구한 뒤 실행하세요.

일반적으로 <code>X</code>가 <code>(n, d)</code>, <code>W</code>가 <code>(d, m)</code>이면 결과 <code>Y</code>는 <code>(n, m)</code>입니다. <code>Y[i, j] = Σₖ X[i, k] W[k, j]</code>에서 Σₖ는 k가 0부터 d−1까지 변할 때 각 곱을 모두 더한다는 기호입니다. k는 입력 특성, i는 이미지, j는 출력 항목을 가리킵니다. 같은 입력 특성끼리 맞춰 더하므로 결과에는 이미지 축과 출력 항목 축이 남습니다. 

**선형 변환** T는 <code>T(u + v) = T(u) + T(v)</code>와 <code>T(a u) = a T(u)</code>를 만족합니다. 여기서 u와 v는 길이가 같은 실수 벡터, a는 실수입니다. 회색조 예제는 실수 벡터에 정의한 이 변환을 0–255 범위의 밝기 데이터에 적용합니다. <code>T(x) = x @ W</code>에서는 각 출력이 입력 좌표들의 가중합이므로 두 식이 곱셈의 분배법칙에서 따릅니다. 예를 들어 [20,80]의 두 밝기를 모두 두 배로 바꾸면 평균 50과 밝기 차 60도 각각 100과 120이 됩니다.


In [3]:
W = np.array([[0.5, -1], [0.5, 1]])
Y = X @ W
print("X, W, Y의 shape:", X.shape, W.shape, Y.shape)
for image_id, (mean, difference) in zip(image_ids, Y):
    print("ID", image_id, "/ 평균", mean, "/ 오른쪽 − 왼쪽", difference)
assert np.array_equal(Y, [[50, 60], [45, -30], [80, 80]])


X, W, Y의 shape: (3, 2) (2, 2) (3, 2)
ID 17 / 평균 50.0 / 오른쪽 − 왼쪽 60.0
ID 52 / 평균 45.0 / 오른쪽 − 왼쪽 -30.0
ID 8 / 평균 80.0 / 오른쪽 − 왼쪽 80.0


### 차원이 맞는지와 특성 순서가 맞는지 확인하기

<code>X @ X</code>에서 첫 X의 열 수는 2이고 둘째 X의 행 수는 3이므로 행렬곱을 할 수 없습니다. 반면 <code>X[:, ::-1] @ W</code>는 실행됩니다. <code>::-1</code>은 순서를 뒤집으므로 이 식은 픽셀 열만 바꾸고 계수 행은 그대로 둡니다.

첫 이미지의 평균은 두 입력의 순서를 바꾸어도 같습니다. 밝기 차는 어떻게 될까요? 픽셀 열과 W의 대응하는 행을 **함께** 바꾸었을 때와 비교해 보세요.


In [4]:
reversed_pixels = X[:, ::-1]
print("픽셀 열만 뒤집은 첫 결과:", (reversed_pixels @ W)[0])
print("계수의 행도 뒤집은 첫 결과:", (reversed_pixels @ W[::-1])[0])
assert np.array_equal(reversed_pixels @ W[::-1], Y)
try:
    X @ X
except ValueError:
    print("X @ X의 가운데 길이:", X.shape[1], X.shape[0])


픽셀 열만 뒤집은 첫 결과: [ 50. -60.]
계수의 행도 뒤집은 첫 결과: [50. 60.]
X @ X의 가운데 길이: 2 3


## 3. 원소별 곱과 행렬곱

<code>A = X[:2]</code>는 처음 두 이미지의 픽셀 배열로 shape가 <code>(2, 2)</code>입니다. 단위행렬 <code>I = [[1, 0], [0, 1]]</code>의 첫 열은 왼쪽 픽셀, 둘째 열은 오른쪽 픽셀을 그대로 골라냅니다.

<code>A * I</code>와 <code>A @ I</code>의 shape는 둘 다 <code>(2, 2)</code>입니다. 두 결과의 **[1, 0]**에 들어갈 값을 각각 쓰세요. 식에서 쓰는 A의 행과 I의 열을 표시해 설명하세요. 대각선 두 칸만 비교하면 두 연산을 구별할 수 있을까요?


In [5]:
A = X[:2]
identity = np.eye(2, dtype=int)
by_position = A * identity
by_rule = A @ identity
print("원소별 곱:\n", by_position)
print("행렬곱:\n", by_rule)
print("[1, 0] 비교:", by_position[1, 0], by_rule[1, 0])
assert by_position[1, 0] == 0 and by_rule[1, 0] == 60


원소별 곱:
 [[20  0]
 [ 0 30]]
행렬곱:
 [[20 80]
 [60 30]]
[1, 0] 비교: 0 60


## 4. 배열의 결합

후보 벡터를 두 번에 나누어 읽었습니다. 각 입력 배열은 <code>(2, 3)</code>이고, 행은 후보, 열은 특성입니다. 첫 배열의 후보 ID는 F1·F2, 둘째 배열의 후보 ID는 S1·S2입니다.

| 연산 | 결과 shape | 축의 의미 |
|---|---|---|
| <code>stack(..., axis=0)</code> | (2, 2, 3) | 입력 묶음 × 묶음 안의 후보 × 특성 |
| <code>concatenate(..., axis=0)</code> | (4, 3) | 후보 × 특성 |
| <code>concatenate(..., axis=1)</code> | (2, 6) | 행 위치 × 옆으로 이어 붙인 값 |

<code>stack</code>은 같은 모양의 배열들을 새 축에 쌓습니다. <code>concatenate</code>는 기존 축을 따라 이어 붙이므로 그 축 이외의 길이가 같아야 합니다. 마지막 행의 방법에서는 F1과 S1의 좌표가 결과의 첫 행에 함께 들어갑니다. 한 후보의 특성을 늘리는 경우라면 두 배열의 행을 먼저 같은 후보 ID에 맞춰야 합니다.


In [6]:
import numpy as np

first = np.array([[1, 0, 2], [0, 1, 1]])
second = np.array([[2, 0, 0], [0, 2, 1]])
bundled = np.stack([first, second], axis=0)
joined_rows = np.concatenate([first, second], axis=0)
joined_columns = np.concatenate([first, second], axis=1)
print("새 묶음 축:", bundled.shape)
print("기존 행 축 연결:", joined_rows.shape)
print("기존 열 축 연결:", joined_columns.shape)
print("첫 묶음의 첫 후보:", bundled[0, 0])
try:
    np.stack([first[:1], second], axis=0)
except ValueError as error:
    print("서로 다른 모양을 stack:", type(error).__name__)
try:
    np.concatenate([first, second[:, :2]], axis=0)
except ValueError as error:
    print("잇지 않는 열의 길이가 다름:", type(error).__name__)
assert (bundled.shape, joined_rows.shape, joined_columns.shape) == ((2, 2, 3), (4, 3), (2, 6))


새 묶음 축: (2, 2, 3)
기존 행 축 연결: (4, 3)
기존 열 축 연결: (2, 6)
첫 묶음의 첫 후보: [1 0 2]
서로 다른 모양을 stack: ValueError
잇지 않는 열의 길이가 다름: ValueError


<a id="bridge-scores"></a>

## 5. 질의와 후보의 점수 행렬

**질의(query)**는 검색할 짝을 찾기 위해 주어진 입력이고, **후보(candidate)**는 그 짝으로 선택할 수 있는 대상입니다. **후보 ID**는 후보를 구별하는 식별자입니다. 후보 ID는 그 후보가 현재 목록의 몇 번째에 놓였는지와 관계없이 같은 대상을 가리킵니다.

IOAI 2025 *Restroom Icon Matching*의 질의와 후보는 화장실 표지 이미지입니다. 두 이미지 목록에는 각각 별도로 번호가 붙어 있습니다. 질의마다 같은 화장실의 반대 성별 표지에 해당하는 **후보 이미지 ID**를 출력합니다. 원문의 baseline은 이미지 인코더로 각 이미지를 벡터로 바꾼 뒤 질의와 후보의 점수를 계산합니다. [공식 문제 2–6절](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Restroom/Restroom.ipynb)

여기서는 점수 계산만 다루기 위해 **직접 만든 2차원 벡터**를 입력으로 사용합니다. Q의 각 행은 질의, G의 각 행은 후보입니다. 두 배열의 같은 열은 같은 좌표입니다. 이 절의 첫 코드 셀에서 NumPy, Q, G를 준비하므로 새 커널에서 바로 시작할 수 있습니다.

| 배열 | shape | 행 × 열 |
|---|---|---|
| Q | (2, 2) | 질의 × 좌표 |
| G | (4, 2) | 후보 × 좌표 |
| G.T | (2, 4) | 좌표 × 후보 |
| S = Q @ G.T | (2, 4) | 질의 × 후보 |

후보 j의 벡터는 G의 j번 행입니다. <code>G.T</code>는 그 벡터를 j번 **열**에 놓습니다. 따라서 <code>S[i, j]</code>는 <code>Q[i]</code>와 <code>G[j]</code>의 내적입니다. 질의 두 개와 후보 네 개를 각각 짝지어 여덟 개의 점수를 얻습니다.

**원시 내적**은 길이를 정규화하기 전, 입력 벡터의 좌표 그대로 구한 내적입니다. 첫 질의 [1,0]과 후보 42=[3,4]의 원시 내적은 3, 후보 19=[2,0]과의 내적은 2입니다. 내적은 좌표 크기의 영향을 받습니다. 같은 입력에서 **거리**를 재면 어떨까요? 후보 42·19·35=[1,0.5]까지의 제곱거리는 각 좌표 차의 제곱합이므로 20, 1, 0.25입니다. 가까운 후보를 고를 때는 작은 값을 택합니다.


In [7]:
import numpy as np

query_ids = np.array([103, 109])
candidate_ids = np.array([42, 7, 19, 35])
Q = np.array([[1, 0], [0, 1]])
G = np.array([[3, 4], [0, 2], [2, 0], [1, 0.5]])

raw_scores = Q @ G.T
print("Q/G shape:", Q.shape, G.shape)
print("원시 내적 점수표:\n", raw_scores)
print("원시 내적 최댓값의 후보 ID:", candidate_ids[np.argmax(raw_scores, axis=1)])
print("후보 행별 L2 길이:", np.linalg.norm(G, axis=1))
assert np.array_equal(raw_scores, [[3, 0, 2, 1], [4, 2, 0, 0.5]])


Q/G shape: (2, 2) (4, 2)
원시 내적 점수표:
 [[3.  0.  2.  1. ]
 [4.  2.  0.  0.5]]
원시 내적 최댓값의 후보 ID: [42 42]
후보 행별 L2 길이: [5.         2.         2.         1.11803399]


### 코사인 유사도와 행별 정규화

벡터의 **길이**는 각 좌표 제곱의 합에 제곱근을 취한 값입니다. [3,4]의 길이는 5이므로 길이로 나누면 [3/5,4/5]가 됩니다. [2,0]도 같은 방법으로 나누면 [1,0]입니다. 각 벡터를 자기 길이로 나누어 길이 1로 만드는 연산을 여기서는 **행별 정규화**라고 합니다.

0이 아닌 두 벡터를 각각 길이 1로 만든 뒤 구한 내적이 **코사인 유사도**입니다. <code>cos(q,g) = (q·g) / (||q|| × ||g||)</code>에서 <code>||v||</code>는 v의 길이입니다. 코사인은 같은 방향이면 1, 직각이면 0, 반대 방향이면 −1입니다. 질의 [1,0]과 후보 42의 코사인은 3/5, 후보 19와는 1입니다. 원시 내적과 코사인이 각각 어느 후보에 더 높은 점수를 주는지 비교하세요.

<code>np.linalg.norm(G, axis=1, keepdims=True)</code>는 후보 한 행 안의 좌표들을 사용해 각 후보의 길이 하나를 구합니다. <code>(4,1)</code> 모양을 유지하므로 각 행의 모든 좌표를 그 행의 길이로 나눌 수 있습니다. 0벡터는 분모가 0이므로 코사인이 정의되지 않습니다. 아래 함수는 유한한 수로 이루어진 작은 2차원 배열을 입력으로 받고, 계산할 수 없는 길이는 오류로 알립니다.


In [8]:
def normalize_rows(vectors):
    rows = np.asarray(vectors)
    if rows.ndim != 2 or 0 in rows.shape:
        raise ValueError("비어 있지 않은 2차원 배열이 필요합니다")
    if rows.dtype.kind not in "iuf" or not np.isfinite(rows).all():
        raise ValueError("좌표는 유한한 정수 또는 실수여야 합니다")
    with np.errstate(over="ignore", under="ignore", invalid="ignore"):
        rows = rows.astype(float, copy=False)
        lengths = np.linalg.norm(rows, axis=1, keepdims=True)
    if not np.isfinite(lengths).all() or (lengths == 0).any():
        raise ValueError("각 행의 길이는 계산 가능한 양수여야 합니다")
    return rows / lengths

def cosine_scores(queries, candidates):
    query_unit = normalize_rows(queries)
    candidate_unit = normalize_rows(candidates)
    if query_unit.shape[1] != candidate_unit.shape[1]:
        raise ValueError("질의와 후보의 특성 수가 다릅니다")
    return query_unit @ candidate_unit.T

Q_unit = normalize_rows(Q)
G_unit = normalize_rows(G)
scores = cosine_scores(Q, G)
positions = np.argmax(scores, axis=1)
matched_ids = candidate_ids[positions]
print("길이 분모 shape:", np.linalg.norm(Q, axis=1, keepdims=True).shape,
      np.linalg.norm(G, axis=1, keepdims=True).shape)
print("각 후보 정규화 후 길이:", np.linalg.norm(G_unit, axis=1))
print("코사인 점수표:", np.round(scores, 3), "/ shape:", scores.shape)
print("질의 ID → 코사인 최댓값 후보 ID:",
      dict(zip(query_ids.tolist(), matched_ids.tolist())))
assert scores.shape == (2, 4) and np.allclose(np.linalg.norm(G_unit, axis=1), 1)
assert np.array_equal(matched_ids, [19, 7])


길이 분모 shape: (2, 1) (4, 1)
각 후보 정규화 후 길이: [1. 1. 1. 1.]
코사인 점수표: [[0.6   0.    1.    0.894]
 [0.8   1.    0.    0.447]] / shape: (2, 4)
질의 ID → 코사인 최댓값 후보 ID: {103: 19, 109: 7}


### 좌표의 배율과 동점

벡터 전체에 같은 양수를 곱하면 길이도 그 배수만큼 커집니다. 나눈 뒤의 좌표와 코사인은 같습니다. 한 벡터에 음수를 곱하면 코사인의 부호가 바뀝니다. 각 **열**을 서로 다른 수로 나누면 한 벡터의 좌표 비율도 달라질 수 있습니다. 아래 코드에서 열별로 나눈 후보들의 행 길이를 확인하세요.

이 예제에서는 최고점이 둘 이상이면 현재 후보 순서에서 먼저 놓인 것을 선택합니다. <code>np.argmax(scores, axis=1)</code>은 질의마다 후보 열을 비교해 첫 최고점의 **열 위치**를 돌려줍니다. 같은 방향의 후보 둘을 함께 뒤집었을 때 최종 ID가 바뀌는 이유를 설명해 보세요.


In [9]:
wrong_axis = G / np.linalg.norm(G, axis=0, keepdims=True)
print("열별로 잘못 나눈 shape / 행 길이:",
      wrong_axis.shape, np.round(np.linalg.norm(wrong_axis, axis=1), 3))
print("양수 배율의 점수 불변:", np.allclose(cosine_scores(3 * Q, G), scores))
print("한 질의에 음수를 곱한 점수:", np.round(cosine_scores(-Q[:1], G), 3))
assert np.allclose(cosine_scores(-Q[:1], G), -scores[:1])

tie_ids = np.array([19, 8])
tie_scores = cosine_scores([[1, 0]], [[2, 0], [4, 0]])[0]
print("동률 점수 / 현재 첫 ID:", tie_scores, tie_ids[np.argmax(tie_scores)])
print("동률 행·ID를 함께 뒤집은 첫 ID:", tie_ids[::-1][np.argmax(tie_scores[::-1])])
for invalid in ([[0, 0]], [[np.nan, 1]], [[np.inf, 1]]):
    try:
        normalize_rows(invalid)
    except ValueError as error:
        print("입력 거부:", error)


열별로 잘못 나눈 shape / 행 길이: (4, 2) [1.197 0.444 0.535 0.289]
양수 배율의 점수 불변: True
한 질의에 음수를 곱한 점수: [[-0.6    0.    -1.    -0.894]]
동률 점수 / 현재 첫 ID: [1. 1.] 19
동률 행·ID를 함께 뒤집은 첫 ID: 8
입력 거부: 각 행의 길이는 계산 가능한 양수여야 합니다
입력 거부: 좌표는 유한한 정수 또는 실수여야 합니다
입력 거부: 좌표는 유한한 정수 또는 실수여야 합니다


### 점수, 열 위치, 후보 ID

첫 질의의 최고 코사인은 1이고, 이 값은 점수표의 2번 열에 있습니다. 제출할 후보 ID는 <code>candidate_ids[2]</code>에 있는 19입니다. 둘째 질의에 대한 답은 1번 열의 ID 7입니다. 이 예제에서는 각 질의의 최고점이 하나씩입니다.

후보 행을 재배열하면 점수표에서는 **후보에 해당하는 열**이 같은 순서로 움직입니다. ID 배열도 함께 옮겨야 각 열이 원래의 대상을 가리킵니다. 아래에서 ID 배열을 그대로 둔 경우와 함께 옮긴 경우를 비교하세요.


In [10]:
order = np.array([2, 0, 3, 1])
G_shuffled = G[order]
correct_ids = candidate_ids[order]
new_positions = np.argmax(cosine_scores(Q, G_shuffled), axis=1)
right = correct_ids[new_positions]
wrong = candidate_ids[new_positions]  # 행만 섞고 ID는 그대로 둔 실수
print("이전/새 G shape:", G.shape, G_shuffled.shape)
print("새 점수의 열 위치:", new_positions)
print("ID도 같이 이동:", right)
print("ID를 그대로 둔 오류:", wrong)
assert np.array_equal(right, [19, 7])
assert np.array_equal(wrong, [42, 35])


이전/새 G shape: (4, 2) (4, 2)
새 점수의 열 위치: [0 3]
ID도 같이 이동: [19  7]
ID를 그대로 둔 오류: [42 35]


### 토큰 ID로 임베딩 표 조회하기

**임베딩**은 단어나 이미지 같은 대상을 수치 벡터로 나타낸 표현입니다. IOAI 2026 *Potato*는 단어 1,602개와 <code>(1602, 2560)</code> 모양의 공개 임베딩 배열을 제공합니다. 어휘 목록의 i번째 단어에 배열의 i번째 행이 대응합니다. 심판은 별도의 비공개 표현으로 단어들을 비교합니다. [공식 데이터 명세](https://github.com/IOAI-official/IOAI-2026/blob/e1963801aa4cf33a5a03520b39fa6620a7894c54/Individual-Contest/3_Potato/statement.md#dataset)

다음은 행 6개, 좌표 3개로 **별도로 만든 표 E**입니다. 문장은 단어나 단어 조각을 나타내는 **토큰**의 ID 목록으로 입력합니다. 이 표에서 토큰 ID는 행 번호입니다. 짧은 목록에는 빈자리를 표시하는 PAD(ID 0)를 넣습니다. PAD 행은 [8,8,8]이고 실제 토큰 ID 5의 행은 [0,0,0]입니다. 유효한 자리는 토큰 ID로 판별합니다.

어휘 수가 V, 벡터 차원이 d이면 E의 shape는 <code>(V,d)</code>입니다. B개 문장을 각각 T칸에 담은 ID 배열 I의 shape는 <code>(B,T)</code>이고, <code>H = E[I]</code>의 shape는 <code>(B,T,d)</code>입니다. 한 좌표를 쓰면 <code>H[b,t,j] = E[I[b,t], j]</code>입니다. I에서 얻은 수로 E의 행을 고르고, 그 행의 j번 좌표를 읽습니다.

아래 둘째 문장에서 같은 토큰이 반복되는 위치를 찾으세요. 외부 파일의 ID 41에는 이 표의 3번 행이 대응한다고 따로 지정했습니다. 외부 ID를 곧바로 배열 인덱스로 써도 되는지 그 예에서 확인하세요.


In [11]:
embedding_table = np.array([
    [8, 8, 8],  # 이 예의 PAD 행: 일부러 0이 아님
    [2, 0, 1], [0, 2, 1], [2, 2, 0],
    [5, -1, 3], [0, 0, 0],  # ID 5는 유효한 실제 토큰
], dtype=float)
token_indices = np.array([[1, 2, 0, 0], [3, 5, 3, 0]])
looked_up = embedding_table[token_indices]  # 예제 ID는 모두 범위 안
external_id_to_row = {41: 3, 87: 1}
print("표 / 토큰 ID / 조회 shape:",
      embedding_table.shape, token_indices.shape, looked_up.shape)
print("조회한 벡터:\n", looked_up)
print("외부 ID 41 → 표 행:", external_id_to_row[41],
      "→", embedding_table[external_id_to_row[41]])
assert looked_up.shape == (2, 4, 3)
assert np.array_equal(looked_up[1, 0], looked_up[1, 2])


표 / 토큰 ID / 조회 shape: (6, 3) (2, 4) (2, 4, 3)
조회한 벡터:
 [[[2. 0. 1.]
  [0. 2. 1.]
  [8. 8. 8.]
  [8. 8. 8.]]

 [[2. 2. 0.]
  [0. 0. 0.]
  [2. 2. 0.]
  [8. 8. 8.]]]
외부 ID 41 → 표 행: 3 → [2. 2. 0.]


### 유효한 토큰의 좌표별 평균

첫 문장의 실제 토큰 벡터는 [2,0,1], [0,2,1]입니다. 좌표별 합 [2,2,2]를 유효 토큰 수 2로 나누면 평균 [1,1,1]입니다. 둘째 문장에서는 ID 3이 두 번, ID 5가 한 번 나옵니다. [0,0,0]인 실제 토큰도 개수에 포함됩니다.

문장 b의 위치 t가 유효하면 <code>m[b,t]=1</code>, PAD이면 0입니다. 유효 개수는 <code>Σₜ m[b,t]</code>, j번째 좌표의 평균은 <code>Σₜ (m[b,t] × H[b,t,j]) / 유효 개수</code>입니다. 합을 구하는 축은 **문장 안의 자리 t**입니다. 결과에는 문장 축 b와 좌표 축 j가 남습니다.

코드의 <code>valid = ids != 0</code>은 (B,T) 모양의 참·거짓 배열입니다. <code>valid[:, :, None]</code>으로 (B,T,1)을 만들면 한 토큰의 모든 좌표에 같은 조건을 적용합니다. <code>np.where</code>로 PAD 위치의 값을 0으로 바꾼 뒤 <code>sum(axis=1)</code>을 계산합니다. 합과 평균은 (B,d), 분모는 (B,1)입니다.

모든 자리가 PAD이면 분모가 0이어서 평균이 없습니다. 실제 ID 5 하나는 개수가 1이고 평균도 정의됩니다. 아래 함수는 음수·범위 밖·정수가 아닌 ID를 조회 전에 검사합니다. NumPy에서 −1은 마지막 행을 가리키기 때문입니다.


In [12]:
def lookup_and_masked_mean(table, token_ids):
    table = np.asarray(table)
    ids = np.asarray(token_ids, dtype=object)  # True가 정수 ID로 바뀌지 않게 확인
    if (table.ndim != 2 or 0 in table.shape or table.dtype.kind not in "iuf"
            or not np.isfinite(table).all()):
        raise ValueError("표는 비어 있지 않은 유한 실수 배열이어야 합니다")
    if ids.ndim != 2 or 0 in ids.shape:
        raise ValueError("ID는 비어 있지 않은 2차원 배열이어야 합니다")
    if any(isinstance(i, (bool, np.bool_)) or not isinstance(i, (int, np.integer))
           for i in ids.flat):
        raise ValueError("토큰 ID는 정수여야 합니다")
    if any(i < 0 or i >= table.shape[0] for i in ids.flat):
        raise ValueError("토큰 ID가 어휘표 범위를 벗어났습니다")
    ids = ids.astype(np.intp)
    valid = ids != 0                 # 이 예의 PAD ID
    counts = valid.sum(axis=1, keepdims=True)
    if (counts == 0).any():
        raise ValueError("전부 PAD이면 평균이 없습니다")
    vectors = table[ids]
    with np.errstate(over="ignore", invalid="ignore"):
        sums = np.where(valid[:, :, None], vectors, 0).sum(axis=1, dtype=float)
        means = sums / counts
    if not np.isfinite(means).all():
        raise ValueError("유한한 평균을 계산할 수 없습니다")
    return vectors, valid, sums, counts, means

vectors, valid, sums, counts, means = lookup_and_masked_mean(
    embedding_table, token_indices)
print("유효 위치:\n", valid)
print("유효 합 / 개수:\n", sums, "\n", counts)
print("문장별 평균:", means)
print("잘못 전체 네 칸으로 평균:", vectors.mean(axis=1))
reversed_vectors, _, _, _, reversed_means = lookup_and_masked_mean(
    embedding_table, token_indices[:, ::-1])
print("순서를 바꾸면 조회 자리는 같은가요?", np.array_equal(vectors, reversed_vectors))
print("순서를 바꾸어도 평균은 같은가요?", np.allclose(means, reversed_means))
assert counts.ravel().tolist() == [2, 3]
assert np.allclose(means, [[1, 1, 1], [4/3, 4/3, 0]])
assert not np.array_equal(vectors, reversed_vectors)
assert np.allclose(means, reversed_means)


유효 위치:
 [[ True  True False False]
 [ True  True  True False]]
유효 합 / 개수:
 [[2. 2. 2.]
 [4. 4. 0.]] 
 [[2]
 [3]]
문장별 평균: [[1.         1.         1.        ]
 [1.33333333 1.33333333 0.        ]]
잘못 전체 네 칸으로 평균: [[4.5 4.5 4.5]
 [3.  3.  2. ]]
순서를 바꾸면 조회 자리는 같은가요? False
순서를 바꾸어도 평균은 같은가요? True


토큰 ID [1,2]와 [2,1]을 조회하면 첫 자리에 오는 벡터가 서로 다릅니다. 고정된 두 벡터의 합과 개수는 어느 순서에서도 같으므로 평균은 같습니다. 평균만 저장한 뒤 원래의 첫 토큰을 결정할 수는 없습니다.

IOAI 2026 *Find the Order*는 대화의 음성 조각들을 받아, 출력의 i번째 칸에 **조각 i의 시간 순위**를 쓰도록 합니다. 순서를 맞혀야 하는 문제에서는 그 순서를 판별할 정보를 보존해야 합니다. 이 절에서는 고정 벡터의 단순 평균에서 사라지는 정보를 확인했습니다. [공식 출력 명세](https://github.com/IOAI-official/IOAI-2026/blob/e1963801aa4cf33a5a03520b39fa6620a7894c54/Individual-Contest/1_Find_the_Order/statement.md#output)


## 6. 연습문제

### A. RGB 채널의 선형 변환

입력 배열의 한 행에는 한 이미지에서 선택한 픽셀의 세 채널 밝기가 들어 있습니다. 채널 이름 R, G, B는 빨강, 초록, 파랑을 뜻하며 값의 범위는 각각 0–255입니다. 이번 입력의 **열 순서는 B, R, G**입니다. 아래 ID와 값은 연습용입니다. 각 이미지 ID에 대해 <code>R − B</code>와 <code>(R + G + B)/3</code>을 이 순서로 출력하세요. 둘째 값은 세 채널의 산술평균입니다.

1. 세 이미지의 답을 손으로 구하고, 그중 한 행에서 두 식에 어떤 열을 사용했는지 표시하세요. 세 입력 채널에서 두 출력을 한꺼번에 구하는 계수 행렬을 직접 만들고, 입력·계수·출력의 shape를 쓰세요.
2. 코드로 <code>(이미지 ID, R−B, 채널 평균)</code>을 입력 행 순서대로 구하세요. 입력의 열 이름과 값이 함께 재배열되어도 같은 결과가 나와야 합니다. 반대로 값만 옮겨 이름이 잘못 붙은 입력을 하나 만들어, 두 출력 중 어느 값에서 오류가 드러나는지 보여 주세요.
3. 세 채널 중 G가 0인 경우와 G 열이 빠진 경우를 비교하세요. 어느 식을 계산할 수 있나요? 이 문제는 두 출력을 모두 요구하므로, 열 이름이 빠지거나 같은 이름이 중복되면 입력 오류로 처리하세요. 밝기 차가 음수인 예도 확인하세요.


In [13]:
import numpy as np

image_ids_new = np.array([31, 7, 19])
channel_names = ["B", "R", "G"]
X_new = np.array([[2, 14, 8], [12, 3, 6], [4, 9, 2]], dtype=np.uint8)
# 원본 픽셀은 부호 없는 8비트 정수입니다. 뺄셈 전에 부호 있는 수로 바꿀 수 있습니다.


손계산한 두 특성과 계수 행렬의 행·열

열 이름과 값이 함께 바뀐 경우의 결과, 값만 옮긴 경우에 달라진 값

G가 0일 때와 G 열이 없을 때 각각 계산할 수 있는 값


In [14]:
# A. RGB 특성을 계산하고 열 순서를 바꾼 입력으로 확인하세요.


### B. 토큰 평균과 후보 검색

입력은 토큰 벡터 표, 질의마다 저장한 토큰 ID, 후보 벡터와 각 ID입니다. 질의 벡터는 실제 토큰들의 좌표별 평균입니다. ID 0은 PAD이며 ID 4는 실제 토큰입니다. 질의와 후보는 같은 두 좌표를 씁니다.

점수 함수는 **원시 내적**과 **코사인 유사도** 두 가지입니다. 다음 두 조건에 해당하는 함수를 각각 정하세요.

| 조건 | 점수의 성질 |
|---|---|
| ① | 두 벡터의 방향이 가까울수록 높은 점수를 줍니다. 어느 한 벡터 전체를 양수 배로 늘려도 점수가 같습니다. |
| ② | 대응하는 좌표끼리 곱한 값들의 합이 클수록 높은 점수를 줍니다. 후보 벡터의 크기도 점수에 반영합니다. |

두 조건 모두 **소수점 아래 여섯 자리로 반올림한 점수**를 비교합니다. 최고점이 같으면 후보 ID가 작은 쪽을 선택합니다. 출력 목록에는 요청서의 질의 ID 순서대로 후보 ID를 하나씩 담습니다.

1. 각 질의의 유효 토큰 수, 좌표별 합과 평균을 구하세요. 점수표의 두 축을 적고, 두 검색의 출력 목록을 먼저 손으로 구한 뒤 코드로 확인하세요. 첫 출력값을 만든 질의 행, 후보 열 위치, 후보 ID를 구별하세요.
2. 후보표 맨 뒤에 후보 ID **6**, 벡터 **[0.5, 0.5]**인 행을 추가합니다. 두 출력 중 어느 것이 달라질까요? 후보 행과 ID를 함께 뒤집은 뒤에도 같은 코드로 확인하세요.
3. 질의별 평균만 저장해 두었습니다. 평균으로 **첫 실제 토큰 ID와 실제 토큰 개수**를 항상 알아낼 수 있는지 판단하세요. 서로 다른 토큰 목록 두 개를 만들어 근거를 보이고, 두 값을 구하려면 무엇을 더 저장해야 하는지 적으세요.

ID 4 하나만 있는 질의와 전부 PAD인 질의도 각각 시험하세요. 어느 경우에 평균, 원시 내적, 코사인을 계산할 수 있나요?


In [15]:
import numpy as np

q_ids_new = np.array([61, 22])         # 문구가 저장된 행의 질의 ID
requested_query_ids = [22, 61]         # 요청서의 답안 순서
g_ids_new = np.array([90, 8, 53, 17])  # 후보표의 행에 붙은 ID
token_table_new = np.array([[7, 11], [2, 0], [2, 9], [6, 4], [0, 0]])
query_tokens_new = np.array([[1, 1, 2, 0], [3, 4, 0, 0]])
gallery_new = np.array([[2, 1], [0, 3], [1, 1], [2, 0]])


조건 ①·②의 답과 한 후보 점수의 계산

후보 행을 추가하거나 옮겼을 때의 예측과 실행 결과

평균으로 복원할 수 있는 정보와 반례, 영벡터·전부 PAD인 입력의 처리


In [16]:
# B. 질의 벡터, 두 종류의 점수와 요청 순서의 후보 ID를 계산하세요.


### C. 두 배열의 결합

첫 배열에는 후보 90=[2,1]과 8=[0,3], 둘째 배열에는 53=[1,1]과 17=[2,0]이 있습니다. 입력 묶음을 구별하는 배열과 전체 후보를 한 표에 모은 배열을 각각 만드세요. 열을 옆으로 이어 붙이면 첫 행에 누구의 좌표들이 들어가나요? 첫 배열에 후보 한 개, 둘째 배열에 세 개가 있을 때도 각 방법이 가능한지 확인하세요.

## 원문 문제 · 벡터 점수와 실제 정답

IOAI 2025 *Restroom Icon Matching* 원문의 정답은 **같은 화장실의 반대 성별 표지**입니다. 아래는 두 조건을 손으로 확인하려고 만든 별도 연습 표입니다. 질의는 A 장소의 남성용 표지입니다.

| 후보 ID | 장소 | 표지 | 연습용 점수 |
|---:|---|---|---:|
| 73 | A | 남성용 | 0.99 |
| 18 | B | 여성용 | 0.97 |
| 40 | A | 여성용 | 0.86 |

원문이 요구한 후보 ID는 어느 것인가요? 가장 높은 점수나 두 번째로 높은 점수만 택한 결과와 비교하세요. 실제 대회에서는 장소·표지 종류를 적어 둔 위 표가 제공되지 않습니다. 훈련 데이터로 모델을 준비하고, 질의 이미지마다 짝인 후보 이미지 ID를 예측합니다. [공식 문제와 baseline](https://github.com/IOAI-official/IOAI-2025/blob/39558e9a639d170bd92de91958fcdc915f670463/Individual-Contest/Restroom/Restroom.ipynb)

배열의 축을 보충했다면 [통합 교안 3절](week01_integrated.ipynb#week01-axes), 검색을 보충했다면 [통합 교안 6절](week01_integrated.ipynb#week01-match)로 돌아갑니다. 자신의 결과는 [통합 교안 8절](week01_integrated.ipynb#week01-check)에서 확인하세요.

## 참고 자료

- [NumPy 행렬곱](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) · [벡터 길이](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) · [최댓값의 위치](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)
- [NumPy stack](https://numpy.org/doc/stable/reference/generated/numpy.stack.html) · [concatenate](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html) · [PyTorch 임베딩 표](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Embedding.html)
- [Python과 표 데이터](week01_python_tables.ipynb) · [라벨이 일부만 있는 데이터의 분류](week01_problem_pipeline.ipynb#pipeline-contract) · [10주 학습 계획](../../curriculum/student-plan.html#week-1)
- 국내 중등부 출제 기준은 2027년 국내 공고에서 연결한 자료를 따릅니다. 2026년 11월 20일 확인한 국내 실라버스의 표기는 IOAI 2026입니다. [KOAI 2027 안내](https://kitpa.org/koai/2027) · [국내 출제 기준](https://kitpa.org/koai/syllabus) · [국제 IOAI 2027 실라버스](https://ioai-official.org/wp-content/uploads/2026/11/IOAI_Syllabus.pdf)
